In [ ]:
import os

# Cambia la working directory alla root del progetto
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../'))
os.chdir(project_root)
print(f"Working directory cambiata a: {os.getcwd()}")

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
import sys
import zipfile
import tempfile
sys.path.append('../../')

from app.core.config import settings, QIP_VALORI_PATH, QIP_MAPPING_PATH, ZONE_OMI_PROVINCIA_TORINO_GEOJSON, IMMOBILI_QUOTAZIONE_PATH, ZONE_GRUPPO_QUOTAZIONI_PATH,MISSING_QUOTAZIONI_IDS_PATH

# Lista per raccogliere i motivi di scarto
motivi_scarto = []

## Caricamento dei Dati

In [ ]:
# Carica i dati
immobili_df = pd.read_parquet(settings.DATASET_FULL)
mapping_df = pd.read_csv(settings.IMMOBILI_MAPPING_PATH, sep=';')
qip_df = pd.read_csv(settings.QIP_VALORI_PATH, sep=';')
qip_mapping_df = pd.read_csv(settings.QIP_MAPPING_PATH, sep=';')

# Carica le zone OMI dal file zip
with tempfile.TemporaryDirectory() as temp_dir:
    with zipfile.ZipFile(settings.ZONE_OMI_PROVINCIA_TORINO_GEOJSON, 'r') as zip_ref:
        zip_ref.extractall(temp_dir)
        omi_gdf = gpd.read_file(f"{temp_dir}/zone_omi_provincia_torino.geojson")

# Estrai comune dal campo Name
omi_gdf['Comune'] = omi_gdf['Name'].apply(lambda x: x.split(' - ')[0].strip().replace('39;', "'"))

# Crea dizionario zona to comune
zona_to_comune = dict(zip(omi_gdf['CODZONA'], omi_gdf['Comune']))

## Creazione dei Dizionari di Mappatura

In [ ]:
# Crea dizionari di mappatura
tipologia_to_gruppo = dict(zip(mapping_df['Tipologia Immobile'], mapping_df['Gruppo Catastale']))
categoria_to_gruppo = dict(zip(qip_mapping_df['Categoria'], qip_mapping_df['Gruppo Catastale']))


# Inverti categoria_to_gruppo per gruppo -> categorie
gruppo_to_categorie = {}
for cat, grp in categoria_to_gruppo.items():
    if grp not in gruppo_to_categorie:
        gruppo_to_categorie[grp] = []
    gruppo_to_categorie[grp].append(cat)

## Funzione per Trovare la Zona OMI

In [ ]:
# Funzione per trovare zona OMI
def find_omi_zone(lat, lon):
    point = Point(lon, lat)
    for idx, row in omi_gdf.iterrows():
        if row['geometry'].contains(point):
            return row['CODZONA'], row['Comune']
    return None, None

## Precalcolo dei Valori Medi

In [ ]:
# Aggiungi colonna Gruppo a qip_df
qip_df['Gruppo'] = qip_df['Descr_Tipologia'].map(categoria_to_gruppo)

# Aggiungi colonna Comune a qip_df
qip_df['Comune'] = qip_df['Zona'].map(zona_to_comune)

# Calcola valore_medio per ogni riga
qip_df['valore_medio'] = (qip_df['Compr_min'] + qip_df['Compr_max']) / 2

# Precalcola i valori medi per comune, zona e gruppo catastale (media dei valori medi)
qip_pivot = qip_df.groupby(['Comune', 'Zona', 'Gruppo']).agg({'valore_medio': 'mean'}).reset_index()

# Aggiungi colonna fascia prendendo solo la lettera della zona
qip_pivot['fascia'] = qip_pivot['Zona'].str[0]

# Aggiungi colonna fascia a qip_df
qip_df['fascia'] = qip_df['Zona'].str[0]

# Precalcola valori medi per comune e zona (media sui gruppi)
zona_df = qip_df.groupby(['Comune', 'Zona']).agg({'valore_medio': 'mean'}).reset_index()

# Precalcola valori medi per comune e fascia (media sui gruppi)
fascia_df = qip_df.groupby(['Comune', 'fascia']).agg({'valore_medio': 'mean'}).reset_index()

# Crea set di gruppi validi (presenti in almeno una zona)
valid_groups = set(qip_pivot['Gruppo'].unique())

# Salva le quotazioni per zona e gruppo in un file
qip_pivot.to_csv(ZONE_GRUPPO_QUOTAZIONI_PATH, sep=';', index=False)
print(f"Quotazioni per comune, zona e gruppo salvate in: {ZONE_GRUPPO_QUOTAZIONI_PATH}")

## Funzione per Calcolare la Quotazione

In [ ]:
# Funzione per calcolare quotazione
def calcola_quotazione(row):
    tipologia = row['tipologia_bene_immobile']
    lat = row['latitudine']
    lon = row['longitudine']
    
    if pd.isna(tipologia) or pd.isna(lat) or pd.isna(lon):
        motivo = "dati mancanti (tipologia, lat, lon o superficie)"
        motivi_scarto.append({'id': row['id'], 'motivo': motivo, 'tipologia': tipologia, 'lat': lat, 'lon': lon})
        return np.nan
    
    gruppo = tipologia_to_gruppo.get(tipologia)
    
    zona, comune = find_omi_zone(lat, lon)
    if not zona or not comune:
        motivo = "zona OMI o comune non trovati"
        motivi_scarto.append({'id': row['id'], 'motivo': motivo, 'tipologia': tipologia, 'lat': lat, 'lon': lon})
        print(f"ID {row['id']}: {motivo} - Tipologia: '{tipologia}', Lat: {lat}, Lon: {lon}")
        return np.nan
    
    fascia = zona[0]
    
    # 1. Corrispondenza comune-zona-gruppo-quotazione
    valore_row = qip_pivot.loc[(qip_pivot['Comune'] == comune) & (qip_pivot['Zona'] == zona) & (qip_pivot['Gruppo'] == gruppo), 'valore_medio']
    if not valore_row.empty:
        valore = valore_row.iloc[0]
        return valore
    else:
        # 2. Se non c'è, comune-fascia-gruppo-quotazione
        valore_row_fascia_gruppo = qip_pivot.loc[(qip_pivot['Comune'] == comune) & (qip_pivot['fascia'] == fascia) & (qip_pivot['Gruppo'] == gruppo), 'valore_medio']
        if not valore_row_fascia_gruppo.empty:
            valore = valore_row_fascia_gruppo.mean()
            return valore
        else:
            # 3. Se non c'è, comune-zona-quotazione (media sui gruppi)
            valore_row_zona = zona_df.loc[(zona_df['Comune'] == comune) & (zona_df['Zona'] == zona), 'valore_medio']
            if not valore_row_zona.empty:
                valore = valore_row_zona.mean()
                return valore
            else:
                # 4. Se non c'è, comune-fascia-quotazione (media sui gruppi)
                valore_row_fascia = fascia_df.loc[(fascia_df['Comune'] == comune) & (fascia_df['fascia'] == fascia), 'valore_medio']
                if not valore_row_fascia.empty:
                    valore = valore_row_fascia.mean()
                    return valore
                else:
                    motivo = "nessuna corrispondenza trovata"
                    motivi_scarto.append({'id': row['id'], 'motivo': motivo, 'comune': comune, 'zona': zona, 'fascia': fascia, 'gruppo': gruppo, 'lat': lat, 'lon': lon})
                    print(f"ID {row['id']}: {motivo} - Comune: '{comune}', Zona: '{zona}', Fascia: '{fascia}', Gruppo: '{gruppo}', Lat: {lat}, Lon: {lon}")
                    return np.nan

## Applicazione della Funzione e Salvataggio

In [ ]:
immobili_df['gruppo_catastale'] = immobili_df['tipologia_bene_immobile'].map(tipologia_to_gruppo)

immobili_df['quotazione_immobiliare_mq'] = immobili_df.apply(calcola_quotazione, axis=1)

# Arrotonda la quotazione al mq all'unità
immobili_df['quotazione_immobiliare_mq'] = immobili_df['quotazione_immobiliare_mq'].round(0)

# Salva il risultato nella cartella META con solo id, gruppo catastale e quotazione
immobili_df[['id', 'gruppo_catastale', 'quotazione_immobiliare_mq']].to_csv(IMMOBILI_QUOTAZIONE_PATH, sep=';', index=False)

print(f"Quotazione immobiliare al mq aggiunta e file salvato in: {IMMOBILI_QUOTAZIONE_PATH}")

# Statistiche sugli scarti
if motivi_scarto:
    scarti_df = pd.DataFrame(motivi_scarto)
    print("\nStatistiche sugli immobili scartati:")
    print(f"Totale immobili scartati: {len(scarti_df)}")
    print("\nConteggio per motivo:")
    print(scarti_df['motivo'].value_counts())
    
    # Salva i dettagli degli scarti in un file
    scarti_df.to_csv(MISSING_QUOTAZIONI_IDS_PATH, sep=';', index=False)
    print(f"\nDettagli scarti salvati in: {MISSING_QUOTAZIONI_IDS_PATH}")
else:
    print("Nessun immobile scartato.")